# Iter-SPIN — Step 2 (batch) + MMLU capability accuracy

Identical to `SPIN_iter_step2.ipynb` (HSIC + WikiText-2 perplexity for every coupled set, one shared
baseline, reversible suppression, bootstrap CIs) **with one addition**: MMLU multiple-choice **accuracy**
per set, so capability is now measured as fluency (perplexity) *and* correctness (accuracy) — matching
SPIN's own capability table (which uses MMLU) and closing the perplexity-only gap in the variants chapters.
MMLU (not MedMCQA) because these chapters are general-domain, not medical. Accuracy is scored by
length-normalised option log-likelihood — no external harness, portable off Kaggle.

Outputs `metrics_batch_{MODEL_TAG}.json` with an added `mmlu` field on the baseline and every set.
Use this for any **future** full Step-2 run; to backfill accuracy onto sets you have ALREADY scored
(without recomputing HSIC/ppl), use the separate gap-filler notebook instead.

## Config

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, re, gc, time, json
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0','sentencepiece'], check=False)
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ---- Hugging Face Authentication Path ----
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF")
login(token=hf_token, add_to_git_credential=False)

# ---- model lineup: uncomment ONE (must match the model whose Iter-SPIN snapshots you're controlling) ----
MODEL_ID='meta-llama/Llama-3.2-3B-Instruct'

USE_FAST=False
HSIC_N=300; PPL_SEQLEN=2048
SIGMAS=[50,100,200,400]
USE_CHAT_TEMPLATE=False
DTYPE=torch.float16
BOOT=500
N_MMLU=1000                                 # seeded MMLU subsample (full 14k is too slow across 13 sets)
SEED=0
MODEL_TAG=MODEL_ID.split('/')[-1]
print('model', MODEL_ID, '| tag', MODEL_TAG, '| sigmas', SIGMAS, '| N_MMLU', N_MMLU)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 87.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


model meta-llama/Llama-3.2-3B-Instruct | tag Llama-3.2-3B-Instruct | sigmas [50, 100, 200, 400] | N_MMLU 1000


## Data + model (once) — adds MMLU sample

In [2]:
if not os.path.exists('DEAN'):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git'], check=True)
DATA='DEAN/src/data'
fair_q=pd.read_csv(f'{DATA}/salad_fairness.csv')['questions'].tolist()[:HSIC_N]
priv_q=pd.read_csv(f'{DATA}/salad_privacy.csv')['questions'].tolist()[:HSIC_N]
_wt=load_dataset('wikitext','wikitext-2-raw-v1',split='test')['text']
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
model.config.use_cache=False  # Gemma-2 hybrid/static KV cache + multi-GPU shard -> cross-device write; forward-only passes don't need it
GEN_DEV=model.get_input_embeddings().weight.device
wt_ids=tokenizer('\n\n'.join(_wt), return_tensors='pt').input_ids
# MMLU (all subjects), seeded subsample
try: _mm=load_dataset('cais/mmlu','all',split='test')
except Exception: _mm=load_dataset('cais/mmlu','all',split='test',trust_remote_code=True)
_mm=_mm.shuffle(seed=SEED).select(range(min(N_MMLU,len(_mm))))
mmlu=[{'prompt':f"Question: {r['question']}\nAnswer:",'choices':[f" {c}" for c in r['choices']],'answer':int(r['answer'])} for r in _mm]
print('model loaded | entry', GEN_DEV, '|', len(fair_q),'fair +',len(priv_q),'priv | MMLU', len(mmlu))

Cloning into 'DEAN'...


Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

model loaded | entry cuda:0 | 300 fair + 300 priv | MMLU 1000


## Metric functions (HSIC/ppl identical to author-sample Step 2; + MMLU accuracy)

In [3]:
def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
@torch.no_grad()
def reps_all(queries,bs=8):
    acc=None
    for i in range(0,len(queries),bs):
        enc=tokenizer(fmt(queries[i:i+bs]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        hs=model(**enc,output_hidden_states=True,use_cache=False).hidden_states
        m=enc.attention_mask.unsqueeze(-1)
        pooled=[((h*m).sum(1)/m.sum(1)).float().cpu().numpy() for h in hs]
        acc=[[p] for p in pooled] if acc is None else [acc[l]+[pooled[l]] for l in range(len(pooled))]
    return [np.concatenate(a,0) for a in acc]
def _d2(A,B):
    sa=(A**2).sum(1,keepdims=True); sb=(B**2).sum(1,keepdims=True); return np.clip(sa+sb.T-2*A@B.T,0,None)
def hsic(X,Y,sigma):
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]
    K=np.exp(-_d2(X,X)/(2*sigma**2)); L=np.exp(-_d2(Y,Y)/(2*sigma**2)); H=np.eye(n)-np.ones((n,n))/n
    return float(np.trace(K@H@L@H)/(n-1)**2)
@torch.no_grad()
def eval_ppl(seqlen=PPL_SEQLEN):
    ns=wt_ids.numel()//seqlen; nlls=[]
    for i in range(ns):
        inp=wt_ids[:,i*seqlen:(i+1)*seqlen].to(GEN_DEV); lg=model(inp,use_cache=False).logits
        nlls.append(torch.nn.CrossEntropyLoss()(lg[:,:-1,:].reshape(-1,lg.size(-1)),inp[:,1:].reshape(-1)).float()*seqlen)
    return torch.exp(torch.stack(nlls).sum()/(ns*seqlen)).item()
@torch.no_grad()
def _choice_loglik(prompt, choice):
    enc=tokenizer(prompt+choice, return_tensors='pt', truncation=True, max_length=1024).to(GEN_DEV)
    plen=tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
    logits=model(**enc,use_cache=False).logits[:, :-1, :]; tgt=enc.input_ids[:, 1:]
    logp=torch.log_softmax(logits, dim=-1).gather(2, tgt.unsqueeze(-1)).squeeze(-1)[0]
    seg=logp[plen-1:]
    return float(seg.mean().item()) if seg.numel()>0 else -1e9
def mc_accuracy(items):
    if not items: return float('nan')
    c=0
    for it in items:
        lls=[_choice_loglik(it['prompt'], ch) for ch in it['choices']]
        c+= int(np.argmax(lls))==it['answer']
    return c/len(items)
def suppress_reversible(coupled):
    d=dict(model.named_parameters()); saved=[]; nz=0
    for n,idxlist in coupled.items():
        if n in d and idxlist:
            p=d[n]; idx=torch.tensor(idxlist,device=p.device); flat=p.data.view(-1)
            saved.append((n,idx,flat[idx].clone())); flat[idx]=0.0; nz+=len(idxlist)
    return saved,nz
def restore(saved):
    d=dict(model.named_parameters())
    for n,idx,vals in saved: d[n].data.view(-1)[idx]=vals
def hsic_pack(Lf,Lp):
    fin={str(s):hsic(Lf[-1],Lp[-1],s) for s in SIGMAS}
    per={str(s):[hsic(Lf[l],Lp[l],s) for l in range(len(Lf))] for s in SIGMAS}
    return fin,per
print('functions ready.')

functions ready.


## Baseline once (HSIC + ppl + MMLU accuracy)

In [4]:
print('baseline reps + ppl + MMLU...')
Lf0=reps_all(fair_q); Lp0=reps_all(priv_q)
base_fin,base_per=hsic_pack(Lf0,Lp0)
Xf0,Xp0=Lf0[-1],Lp0[-1]
del Lf0,Lp0; gc.collect()
base_ppl=eval_ppl(); base_acc=mc_accuracy(mmlu)
print('baseline HSIC(final):',{s:round(base_fin[str(s)],6) for s in SIGMAS},'| ppl',round(base_ppl,4),'| MMLU',round(base_acc,4))

baseline reps + ppl + MMLU...
baseline HSIC(final): {50: 0.000225, 100: 1.8e-05, 200: 1e-06, 400: 0.0} | ppl 11.0469 | MMLU 0.414


## Discover coupled sets

In [5]:
pats={'iter':'coupled_iter_%s_n*.json'%MODEL_TAG,
      'global':'coupled_control_global_%s_n*.json'%MODEL_TAG,
      'percat':'coupled_control_percat_%s_n*.json'%MODEL_TAG}
roots=['.','/kaggle/input','/kaggle/working']
SETS=[]
for variant,pat in pats.items():
    seen=set()
    for r in roots:
        for f in glob.glob(os.path.join(r,'**',pat),recursive=True):
            n=int(re.search(r'_n(\d+)\.json$',f).group(1))
            if (variant,n) in seen: continue
            seen.add((variant,n)); SETS.append((variant,n,f))
SETS.sort(key=lambda x:(x[1],x[0]))
assert SETS, 'no coupled_* JSONs found for %s'%MODEL_TAG
print('found', len(SETS), 'sets')

found 13 sets


## Score every set (suppress -> HSIC + ppl + MMLU -> restore)

In [6]:
rows=[]; REPS={}
for variant,n,f in SETS:
    coupled=json.load(open(f))['coupled']
    saved,nz=suppress_reversible(coupled)
    Lf=reps_all(fair_q); Lp=reps_all(priv_q); fin,per=hsic_pack(Lf,Lp)
    REPS[(variant,n)]=(Lf[-1],Lp[-1]); del Lf,Lp; gc.collect()
    ppl=eval_ppl(); acc=mc_accuracy(mmlu); restore(saved)
    rows.append({'variant':variant,'n':n,'file':os.path.basename(f),'coupled_zeroed':int(nz),
                 'hsic':fin,'hsic_per_layer':per,'ppl':ppl,'mmlu':acc})
    d50=100*(fin['50']-base_fin['50'])/base_fin['50']
    print(f'  {variant:7s} n={n:<3d} zeroed {nz:4d} | HSIC@50 {d50:+.1f}% | ppl {ppl:.3f} | MMLU {acc:.4f} ({100*(acc-base_acc):+.1f}pp)')

out={'model':MODEL_ID,'sigmas':SIGMAS,'n_layers':len(base_per[str(SIGMAS[0])]),'n_mmlu':len(mmlu),
     'baseline':{'hsic':base_fin,'hsic_per_layer':base_per,'ppl':base_ppl,'mmlu':base_acc},'sets':rows}
json.dump(out, open(f'/kaggle/working/metrics_batch_{MODEL_TAG}.json','w'))
print('\nsaved metrics_batch_%s.json (now with MMLU accuracy)'%MODEL_TAG)

  iter    n=1   zeroed  879 | HSIC@50 +4.4% | ppl 11.037 | MMLU 0.4090 (-0.5pp)
  global  n=2   zeroed 1662 | HSIC@50 +10.8% | ppl 11.168 | MMLU 0.4190 (+0.5pp)
  iter    n=2   zeroed 1681 | HSIC@50 +8.8% | ppl 11.024 | MMLU 0.4030 (-1.1pp)
  percat  n=2   zeroed 1681 | HSIC@50 +0.3% | ppl 11.058 | MMLU 0.4120 (-0.2pp)
  global  n=3   zeroed 2413 | HSIC@50 +10.1% | ppl 11.235 | MMLU 0.4190 (+0.5pp)
  iter    n=3   zeroed 2426 | HSIC@50 +14.4% | ppl 11.104 | MMLU 0.4080 (-0.6pp)
  percat  n=3   zeroed 2426 | HSIC@50 +0.3% | ppl 11.067 | MMLU 0.4130 (-0.1pp)
  global  n=5   zeroed 3924 | HSIC@50 +14.3% | ppl 11.264 | MMLU 0.4210 (+0.7pp)
  iter    n=5   zeroed 3915 | HSIC@50 +20.7% | ppl 11.391 | MMLU 0.4010 (-1.3pp)
  percat  n=5   zeroed 3915 | HSIC@50 +0.5% | ppl 11.073 | MMLU 0.4110 (-0.3pp)
  global  n=10  zeroed 7380 | HSIC@50 +12.6% | ppl 11.355 | MMLU 0.4170 (+0.3pp)
  iter    n=10  zeroed 7379 | HSIC@50 +23.9% | ppl 11.981 | MMLU 0.3860 (-2.8pp)
  percat  n=10  zeroed 7379 | HSI

## Matched-budget table (HSIC drop + ppl + MMLU accuracy)

In [7]:
def drop(fin,s): b=base_fin[str(s)]; return 100*(fin[str(s)]-b)/b if b else 0.0
ns=sorted(set(n for _,n,_ in SETS))
print(f'baseline: ' + '  '.join(f'σ{s}={base_fin[str(s)]:.6f}' for s in SIGMAS) + f'  | ppl {base_ppl:.4f} | MMLU {base_acc:.4f}')
hdr=f'{"n":>3} {"variant":>7} {"zeroed":>7} ' + ' '.join(f'{"dHSIC@"+str(s):>10}' for s in SIGMAS) + f' {"dppl%":>7} {"MMLU":>7} {"dMMLU":>7}'
for n in ns:
    print('-'*104); print(hdr); print('-'*104)
    for variant in ['iter','global','percat']:
        r=next((x for x in rows if x['variant']==variant and x['n']==n), None)
        if not r: continue
        cells_=' '.join(f'{drop(r["hsic"],s):>+9.1f}%' for s in SIGMAS)
        print(f'{n:>3} {variant:>7} {r["coupled_zeroed"]:>7} {cells_} {100*(r["ppl"]-base_ppl)/base_ppl:>+6.2f}% {r["mmlu"]:>7.4f} {100*(r["mmlu"]-base_acc):>+6.1f}')

baseline: σ50=0.000225  σ100=0.000018  σ200=0.000001  σ400=0.000000  | ppl 11.0469 | MMLU 0.4140
--------------------------------------------------------------------------------------------------------
  n variant  zeroed   dHSIC@50  dHSIC@100  dHSIC@200  dHSIC@400   dppl%    MMLU   dMMLU
--------------------------------------------------------------------------------------------------------
  1    iter     879      +4.4%      +5.1%      +5.3%      +5.3%  -0.09%  0.4090   -0.5
--------------------------------------------------------------------------------------------------------
  n variant  zeroed   dHSIC@50  dHSIC@100  dHSIC@200  dHSIC@400   dppl%    MMLU   dMMLU
--------------------------------------------------------------------------------------------------------
  2    iter    1681      +8.8%     +10.1%     +10.5%     +10.6%  -0.21%  0.4030   -1.1
  2  global    1662     +10.8%     +12.4%     +12.9%     +13.0%  +1.10%  0.4190   +0.5
  2  percat    1681      +0.3%      +0.3%     

## Bootstrap 95% CIs on HSIC drop (identical to SPIN_iter_step2)

In [8]:
if BOOT and REPS:
    n0=min(len(Xf0),len(Xp0)); rng=np.random.default_rng(0)
    IDX=[rng.integers(0,n0,n0) for _ in range(BOOT)]
    def _kern(X,sig): return np.exp(-_d2(X[:n0],X[:n0])/(2*sig**2))
    def _hsic_idx(Kf,Lp,idx):
        Kb=Kf[np.ix_(idx,idx)]; Lb=Lp[np.ix_(idx,idx)]; n=len(idx)
        Kc=Kb-Kb.mean(0,keepdims=True)-Kb.mean(1,keepdims=True)+Kb.mean()
        return (Kc*Lb).sum()/(n-1)**2
    BK={s:(_kern(Xf0,s),_kern(Xp0,s)) for s in SIGMAS}
    drops={}
    for (variant,n),(Xf,Xp) in REPS.items():
        for s in SIGMAS:
            Kf,Lp=_kern(Xf,s),_kern(Xp,s); Kf0,Lp0=BK[s]; arr=np.empty(BOOT)
            for b,idx in enumerate(IDX):
                base=_hsic_idx(Kf0,Lp0,idx); st=_hsic_idx(Kf,Lp,idx)
                arr[b]=100*(st-base)/base if base else 0.0
            drops[(variant,n,s)]=arr
    ci={f'{v}_n{n}_s{s}':{'med':float(np.median(a)),'lo':float(np.percentile(a,2.5)),'hi':float(np.percentile(a,97.5))} for (v,n,s),a in drops.items()}
    out['bootstrap']={'B':BOOT,'drop_ci':ci}
    json.dump(out, open(f'/kaggle/working/metrics_batch_{MODEL_TAG}.json','w'))
    print('bootstrap CIs added (B=%d).'%BOOT)
else:
    print('BOOT=0 or no sets: skipped.')

bootstrap CIs added (B=500).
